In [7]:
import pandas as pd
import numpy as np
import requests


In [8]:
#Scrape the S&P 500 data from Wikipedia

def get_sp500_data():
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()

    tables = pd.read_html(response.text)
    df = tables[0][['Symbol', 'Security', 'GICS Sector', 'CIK']].copy()

    df['CIK'] = df['CIK'].astype(str).str.zfill(10)
    return df

print(get_sp500_data())

    Symbol             Security             GICS Sector         CIK
0      MMM                   3M             Industrials  0000066740
1      AOS          A. O. Smith             Industrials  0000091142
2      ABT  Abbott Laboratories             Health Care  0000001800
3     ABBV               AbbVie             Health Care  0001551152
4      ACN            Accenture  Information Technology  0001467373
..     ...                  ...                     ...         ...
498    XYL           Xylem Inc.             Industrials  0001524472
499    YUM          Yum! Brands  Consumer Discretionary  0001041061
500   ZBRA   Zebra Technologies  Information Technology  0000877212
501    ZBH        Zimmer Biomet             Health Care  0001136869
502    ZTS               Zoetis             Health Care  0001555280

[503 rows x 4 columns]


C:\Users\adesi\AppData\Local\Temp\ipykernel_15720\3457739051.py:12: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


In [ ]:
import re
from edgar.core import set_identity
from edgar.entity import Company

#Set EDGAR identity for responsible scraping.
set_identity("Emmanuel Adegoke adesina0202@gmail.com")

# Score candidate sections based on length and text density to filter out TOC fragments.
def _score_candidate(chunk: str) -> tuple:
    alpha_chars = sum(ch.isalpha() for ch in chunk)
    digit_chars = sum(ch.isdigit() for ch in chunk)
    alpha_ratio = alpha_chars / max(len(chunk), 1)
    digit_ratio = digit_chars / max(len(chunk), 1)

    # Higher is better: favor longer, text-heavy sections over TOC fragments.
    has_risk_word = 1 if re.search(r"\brisk\b", chunk, flags=re.IGNORECASE) else 0
    return (has_risk_word, alpha_ratio - digit_ratio, len(chunk))

# Enhanced extraction with multiple patterns and scoring to handle formatting variations.
def get_actual_risk_text(ticker: str) -> str:
    filings = Company(ticker).get_filings(form="10-K")
    latest_10k = filings.latest()
    if latest_10k is None:
        return "No 10-K filing found."

    text = latest_10k.text() or ""
    if not text:
        return "No filing text returned."

    # Try multiple boundary patterns because issuers format Item headers differently.
    patterns = [
        re.compile(r"item\s+1a\.?\s*risk\s*factors(.*?)(?:item\s+1b\.|item\s+2\.)", re.IGNORECASE | re.DOTALL),
        re.compile(r"item\s+1a\.?(.*?)(?:item\s+1b\.|item\s+2\.)", re.IGNORECASE | re.DOTALL),
    ]

    candidates = []
    for pattern in patterns:
        for match in pattern.finditer(text):
            chunk = match.group(1).strip()
            if not chunk:
                continue
            # Reject obvious TOC/page-number fragments.
            if re.fullmatch(r"\d{1,4}", chunk):
                continue
            if len(chunk) < 300 and not re.search(r"[A-Za-z]{40,}", chunk):
                continue
            candidates.append(chunk)

    if candidates:
        best = max(candidates, key=_score_candidate)
        return best

    return "Item 1A section not found."

# Clean the extracted text to remove common artifacts and improve readability.
# Common artifacts include decorative rules, page numbers, and filing fragment headers that can interfere with downstream processing.
# This is done to remove noise that could affect embedding quality and to ensure the text is in a more consistent format for analysis.
def clean_risk_text(text: str) -> str:
    # Remove decorative rule characters and separators.
    text = re.sub(r"[\u2500-\u257f]+", " ", text)

    # Remove standalone page numbers at the start or on isolated lines.
    text = re.sub(r"^\s*\d{1,4}\s*$", "", text, flags=re.MULTILINE)

    # Strip common leading filing fragment artifacts.
    text = re.sub(r"^\s*of\s+this\s+(?:annual\s+report\s+on\s+)?form\s+10-k\)\.?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^\s*of\s+part\s+i[,\s“\"']*risk\s*factors[,\s”\"']*under\s+the\s+heading\s*", "", text, flags=re.IGNORECASE)

    # Normalize line endings and trim trailing spaces.
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = "\n".join(line.rstrip() for line in text.split("\n"))

    # Collapse intra-paragraph hard wraps while preserving paragraph breaks.
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)

    # Normalize spacing around punctuation and whitespace runs.
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([,.;:])", r"\1", text)

    # Restore paragraph breaks around common section headers.
    text = re.sub(r"\b(Business and Industry Risks|General Risks|Regulatory Risks)\b", r"\n\n\1", text, flags=re.IGNORECASE)

    return text.strip()

# Split the cleaned text into meaningful blocks based on likely sub-heading boundaries (Title Case phrases).
# Examples include "Business and Industry Risks", "General Risks", "Regulatory Risks", or other Title Case phrases that often denote new sections within the risk factors narrative.
def split_risk_blocks(clean_text: str) -> list[str]:
    # Split on likely sub-heading boundaries (Title Case phrases).
    heading_pattern = re.compile(
        r"(?=(?:[A-Z][A-Za-z&/(),\-]{2,}(?:\s+[A-Z][A-Za-z&/(),\-]{2,}){1,10}))"
    )
    parts = heading_pattern.split(clean_text)

    blocks = []
    current = ""
    for part in parts:
        part = part.strip()
        if not part:
            continue
        if len(part.split()) <= 12 and part == part.title():
            if current:
                blocks.append(current.strip())
            current = part
        else:
            current = f"{current} {part}".strip() if current else part

    if current:
        blocks.append(current.strip())

    # Keep meaningful narrative chunks.
    return [b for b in blocks if len(b) > 180]

# Clean the text further to prepare for embedding by removing common artifacts that could interfere with embedding quality, such as bullet points, list punctuation, and repetitive boilerplate phrases that add little semantic value.
def clean_for_embedding(text: str) -> str:
    # Remove bullet glyphs and list punctuation artifacts.
    text = re.sub(r"[•●◦▪]+", " ", text)

    # Remove repetitive legal boilerplate that usually adds little semantic value.
    text = re.sub(r"\b(the following risk factors do not identify all risks[^.]*\.)", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\breferences to past events[^.]*\.", " ", text, flags=re.IGNORECASE)

    # Normalize again after removals.
    text = re.sub(r"\s+", " ", text).strip()
    return text

def chunk_for_embedding(text: str, target_words: int = 180, overlap_words: int = 40) -> list[str]:
    words = text.split()
    if not words:
        return []

    chunks = []
    i = 0
    step = max(target_words - overlap_words, 1)
    while i < len(words):
        chunk_words = words[i:i + target_words]
        chunk = " ".join(chunk_words).strip()
        if len(chunk) > 120:
            chunks.append(chunk)
        i += step
    return chunks


risk_factors_raw = get_actual_risk_text("AMZN")
risk_factors_clean = clean_risk_text(risk_factors_raw)
risk_blocks = split_risk_blocks(risk_factors_clean)
embedding_ready_text = clean_for_embedding(risk_factors_clean)
embedding_chunks = chunk_for_embedding(embedding_ready_text)

print(risk_factors_clean[:1200])
print(f"\nDetected risk blocks: {len(risk_blocks)}")
print(f"Embedding chunks: {len(embedding_chunks)}")
if embedding_chunks:
    print("\nFirst embedding chunk preview:\n", embedding_chunks[0][:500])

“We Could Be Harmed by Data Loss or Other Security Incidents,” which should be read in conjunction with the information above. The Security Committee, which is comprised of independent directors, oversees our policies and procedures for protecting our cybersecurity infrastructure and for compliance with applicable data protection and security regulations, and related risks. TheSecurity Committeereceives reports regarding such risks from management, including our chief security officer, and reports to the Board at least annually. The Security Committee also oversees the Board’s response to any significant cybersecurity incidents. Our chief security officer, who has extensive cybersecurity knowledge and skills gained from over 15 years of work experience on the security team at Amazon and an extensive career in the technology and cybersecurity industries as a senior executive in the federal government, heads the team responsible for implementing and maintaining cybersecurity and data pro

In [100]:

risk_factors_raw = get_actual_risk_text("IBM")
risk_factors_clean = clean_risk_text(risk_factors_raw)
risk_blocks = split_risk_blocks(risk_factors_clean)
embedding_ready_text = clean_for_embedding(risk_factors_clean)
embedding_chunks = chunk_for_embedding(embedding_ready_text)
print(risk_factors_clean[:1200])
print(f"\nDetected risk blocks: {len(risk_blocks)}")
print(f"Embedding chunks: {len(embedding_chunks)}")
if embedding_chunks:
    print("\nFirst embedding chunk preview:\n", embedding_chunks[0][:1000])

: Risks Related to Our Business Downturn in Economic Environment and Client Spending Budgets Could Impact the Company’s Business: If overall demand for IBM’s products and solutions decreases, whether due to general economic conditions, or a shift in client buying patterns, the company’s revenue and profit could be impacted. Failure of Innovation Initiatives Could Impact the Long-Term Success of the Company: IBM has moved into areas, including those that incorporate or utilize hybrid cloud, AI, quantum and other disruptive technologies, in which it can differentiate itself through responsible innovation, by leveraging its investments in R&D and attracting a successful developer ecosystem. If IBM is unable to continue its cutting-edge innovation in a highly competitive and rapidly evolving environment or is unable to commercialize such innovations, expand and scale them with sufficient speed and versatility or is unable to attract a successful developer ecosystem, the company could fail 

In [ ]:
# Example usage: compare multiple companies
# Microsoft Corporation (MSFT)
# Alphabet Inc. (GOOG)
# Roblox Corporation (RBLX)
# Test the full pipeline on multiple companies to ensure robustness across different risk factor formats and lengths.
def build_company_embedding_view(ticker: str) -> dict:
    raw = get_actual_risk_text(ticker)
    clean = clean_risk_text(raw)
    blocks = split_risk_blocks(clean)
    ready = clean_for_embedding(clean)
    chunks = chunk_for_embedding(ready)
    return {
        "ticker": ticker,
        "clean": clean,
        "blocks": blocks,
        "chunks": chunks,
    }

msft = build_company_embedding_view("MSFT")
goog = build_company_embedding_view("GOOG")
rblx = build_company_embedding_view("RBLX")

for company in [msft, goog, rblx]:
    print(f"\n{company['ticker']} preview:\n", company["clean"][:1000])
    print(f"{company['ticker']} blocks:", len(company["blocks"]))
    print(f"{company['ticker']} embedding chunks:", len(company["chunks"]))
    if company["chunks"]:
        print(f"{company['ticker']} chunk 1:\n", company["chunks"][0][:450])


MSFT preview:
 As of the date of this Form 10-K, we do not believe any risks from cybersecurity threats have materially affected or are reasonably likely to materially affect us, including our results of operations or financial condition. However, the cybersecurity threat environment is increasingly challenging, and we, along with the entire digital ecosystem, are under constant and increasing threat. As discussed above, our business strategy is tied to the SFI and we are committed to continuously monitoring cybersecurity threats, enhancing the security of our products, investing in our cybersecurity infrastructure, and collaborating with peers, customers, service providers, regulators, and governments to advance our and the entire digital ecosystem’s cybersecurity defenses and resiliency. GOVERNANCE Our Board of Directors oversees cybersecurity risk. Cybersecurity reviews by the Board are scheduled to occur at least quarterly, or more frequently as determined to be necessary or advis

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
import time


#Generate dense vector embeddings for the cleaned and chunked risk factor text using a pre-trained Model
#Perform Document-level embedding by averaging chunk embeddings to capture the overall risk profile of the company, which can be used for similarity comparisons across companies or clustering in a vector database.


# all-MiniLM-L6-v2 outputs 384-dimensional dense vectors.
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)


def normalize_ticker(ticker: str) -> str:
    # SEC/EDGAR often uses '-' for class shares (e.g., BRK-B instead of BRK.B).
    return str(ticker).strip().upper().replace(".", "-")

def make_embedding_payload(ticker: str) -> dict:
    raw = get_actual_risk_text(ticker)
    clean = clean_risk_text(raw)
    ready = clean_for_embedding(clean)
    chunks = chunk_for_embedding(ready, target_words=180, overlap_words=40)

    if not chunks:
        return {
            "ticker": ticker,
            "chunks": [],
            "chunk_embeddings": np.array([]),
            "document_embedding": np.array([]),
            "embedding_dim": 0,
            "clean_text": clean,
        }

    chunk_embeddings = model.encode(chunks, normalize_embeddings=True)
    document_embedding = chunk_embeddings.mean(axis=0)

    return {
        "ticker": ticker,
        "chunks": chunks,
        "chunk_embeddings": chunk_embeddings,
        "document_embedding": document_embedding,
        "embedding_dim": int(chunk_embeddings.shape[1]),
        "clean_text": clean,
    }

# ---- Batch extraction for S&P 500 ----
sp500 = get_sp500_data().copy()
sp500["EDGAR_TICKER"] = sp500["Symbol"].apply(normalize_ticker)

MAX_COMPANIES = 500
REQUEST_DELAY_SECONDS = 0.2  # Be polite to upstream services.

success_rows = []
error_rows = []

symbols = sp500["EDGAR_TICKER"].dropna().unique().tolist()[:MAX_COMPANIES]

for i, ticker in enumerate(symbols, start=1):
    try:
        payload = make_embedding_payload(ticker)

        if payload["embedding_dim"] == 0:
            error_rows.append({
                "Ticker": ticker,
                "Reason": "No embedding chunks generated",
            })
        else:
            success_rows.append({
                "Ticker": ticker,
                "EmbeddingDim": payload["embedding_dim"],
                "NumChunks": len(payload["chunks"]),
                "DocumentEmbedding": payload["document_embedding"].tolist(),
                "Preview": payload["chunks"][0][:200],
            })

    except Exception as exc:
        error_rows.append({
            "Ticker": ticker,
            "Reason": str(exc),
        })

    if i % 25 == 0:
        print(f"Processed {i}/{len(symbols)} tickers...")

    time.sleep(REQUEST_DELAY_SECONDS)

embeddings_df = pd.DataFrame(success_rows)
errors_df = pd.DataFrame(error_rows)

# Save outputs for downstream similarity search or vector DB loading.
embeddings_df.to_json("sp500_document_embeddings.json", orient="records")
errors_df.to_csv("sp500_embedding_errors.csv", index=False)

print(f"\nRequested: {len(symbols)} tickers")
print(f"Embedded successfully: {len(embeddings_df)}")
print(f"Failed/empty: {len(errors_df)}")
if not embeddings_df.empty:
    print("\nSample row:")
    print(embeddings_df[["Ticker", "EmbeddingDim", "NumChunks", "Preview"]].head(1))
if not errors_df.empty:
    print("\nSample errors:")
    print(errors_df.head(5))

#This took about 1hour 30 minutes to run for 500 companies, with a 0.2 second delay between requests to be polite to upstream services.

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9363.71it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\adesi\AppData\Local\Temp\ipykernel_14576\2438537438.py:14: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


Processed 25/500 tickers...


Processed 50/500 tickers...


Processed 75/500 tickers...


Processed 100/500 tickers...


...

Processed 400/500 tickers...


Processed 425/500 tickers...


Processed 450/500 tickers...


Processed 475/500 tickers...


Processed 500/500 tickers...

Requested: 500 tickers
Embedded successfully: 420
Failed/empty: 80

Sample row:
  Ticker  EmbeddingDim  NumChunks  \
0    MMM           384         50   

                                             Preview  
0  Provided below is a cautionary discussion of w...  

Sample errors:
  Ticker                         Reason
0    AOS  No embedding chunks generated
1    AMD  No embedding chunks generated
2    ALB  No embedding chunks generated
3    ARE  No embedding chunks generated
4    AEP  No embedding chunks generated


In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

#Load the saved document embeddings from the batch run and prepare them for dimensionality reduction and graph initialization.
#Reduce dimensionality using PCA for later visualisation


# Load saved document embeddings from the batch run.
with open("sp500_document_embeddings.json", "r", encoding="utf-8") as f:
    records = json.load(f)

if not records:
    raise ValueError("sp500_document_embeddings.json is empty.")

embeddings_df = pd.DataFrame(records)
required_cols = {"Ticker", "DocumentEmbedding"}
missing = required_cols - set(embeddings_df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

X = np.array(embeddings_df["DocumentEmbedding"].tolist(), dtype=np.float32)
if X.ndim != 2:
    raise ValueError(f"Expected 2D embedding matrix, got shape {X.shape}")

n_samples, n_features = X.shape
TARGET_DIMS = 32
n_components = min(TARGET_DIMS, n_samples, n_features)
if n_components < TARGET_DIMS:
    print(f"Warning: reduced to {n_components} dims because of matrix shape {X.shape}.")

# Initial state h^(0): PCA-projected dense features.
pca = PCA(n_components=n_components, random_state=42)
h0 = pca.fit_transform(X).astype(np.float32)

h0_df = pd.DataFrame(h0, columns=[f"h0_{i:02d}" for i in range(h0.shape[1])])
h0_df.insert(0, "Ticker", embeddings_df["Ticker"].values)

# Save artifacts for downstream graph initialization.
h0_df.to_csv("sp500_h0_pca32.csv", index=False)
np.save("sp500_h0_pca32.npy", h0)

explained = float(pca.explained_variance_ratio_.sum())
print("Input embedding matrix:", X.shape)
print("h^(0) shape:", h0.shape)
print("Explained variance ratio (sum):", round(explained, 4))
print(h0_df.head(3))

Input embedding matrix: (420, 384)
h^(0) shape: (420, 32)
Explained variance ratio (sum): 0.8306
  Ticker     h0_00     h0_01     h0_02     h0_03     h0_04     h0_05  \
0    MMM  0.163298  0.021371  0.018573 -0.071220  0.141283 -0.071049   
1    ABT -0.498144 -0.060056  0.130288  0.036833  0.041698 -0.050143   
2   ABBV -0.547219 -0.022540 -0.030723  0.031677  0.037050 -0.019598   

      h0_06     h0_07     h0_08  ...     h0_22     h0_23     h0_24     h0_25  \
0 -0.002621  0.081387  0.033542  ...  0.011796 -0.000832  0.014109 -0.047171   
1 -0.081950 -0.022550 -0.066986  ... -0.003176 -0.037783  0.045041 -0.020499   
2  0.092107 -0.050446  0.046648  ...  0.009595  0.012530  0.013583  0.046366   

      h0_26     h0_27     h0_28     h0_29     h0_30     h0_31  
0 -0.014496 -0.004419  0.008422 -0.006417 -0.009473  0.020844  
1 -0.069554 -0.095383  0.046004  0.069123  0.000640  0.038705  
2 -0.062114  0.057737  0.021704  0.023895  0.020580 -0.001085  

[3 rows x 33 columns]


In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

#Construct a k-NN graph based on cosine similarity of the PCA-reduced document embeddings to capture relationships between companies based on their risk factor profiles.


# Load h^(0) features and aligned tickers from the PCA artifact.
h0_df = pd.read_csv("sp500_h0_pca32.csv")
tickers = h0_df["Ticker"].tolist()
X_h0 = h0_df.drop(columns=["Ticker"]).to_numpy(dtype=np.float32)

if X_h0.ndim != 2 or X_h0.shape[0] == 0:
    raise ValueError(f"Invalid h^(0) matrix shape: {X_h0.shape}")

n_nodes = X_h0.shape[0]
K = 10  # Number of nearest neighbors per node.

# k-NN with cosine distance; +1 because each point is its own nearest neighbor.
nbrs = NearestNeighbors(n_neighbors=min(K + 1, n_nodes), metric="cosine", algorithm="brute")
nbrs.fit(X_h0)
distances, indices = nbrs.kneighbors(X_h0)

edge_rows = []
for src in range(n_nodes):
    for dist, dst in zip(distances[src], indices[src]):
        if src == dst:
            continue
        cosine_sim = 1.0 - float(dist)
        edge_rows.append({
            "src_idx": src,
            "dst_idx": int(dst),
            "src_ticker": tickers[src],
            "dst_ticker": tickers[int(dst)],
            "cosine_sim": cosine_sim,
        })

edges_df = pd.DataFrame(edge_rows)

# Symmetrize by keeping the max similarity for each undirected pair.
if not edges_df.empty:
    a = np.minimum(edges_df["src_idx"].to_numpy(), edges_df["dst_idx"].to_numpy())
    b = np.maximum(edges_df["src_idx"].to_numpy(), edges_df["dst_idx"].to_numpy())
    edges_df["u"] = a
    edges_df["v"] = b
    edges_df = edges_df.groupby(["u", "v"], as_index=False).agg({"cosine_sim": "max"})

    undirected_edges = pd.DataFrame({
        "src_idx": np.concatenate([edges_df["u"].to_numpy(), edges_df["v"].to_numpy()]),
        "dst_idx": np.concatenate([edges_df["v"].to_numpy(), edges_df["u"].to_numpy()]),
        "cosine_sim": np.concatenate([edges_df["cosine_sim"].to_numpy(), edges_df["cosine_sim"].to_numpy()]),
    })
else:
    undirected_edges = pd.DataFrame(columns=["src_idx", "dst_idx", "cosine_sim"])

# Save edge artifacts.
undirected_edges.to_csv("sp500_cosine_knn_edges.csv", index=False)
np.save("sp500_cosine_knn_edge_index.npy", undirected_edges[["src_idx", "dst_idx"]].to_numpy(dtype=np.int64).T)
np.save("sp500_cosine_knn_edge_weight.npy", undirected_edges["cosine_sim"].to_numpy(dtype=np.float32))

print("Nodes:", n_nodes)
print("Requested k:", K)
print("Directed edges before symmetrization:", len(edge_rows))
print("Undirected pairs:", len(edges_df) if 'edges_df' in locals() else 0)
print("Directed edges after symmetrization:", len(undirected_edges))
print(undirected_edges.head(10))

Nodes: 420
Requested k: 10
Directed edges before symmetrization: 4200
Undirected pairs: 3031
Directed edges after symmetrization: 6062
   src_idx  dst_idx  cosine_sim
0        0       31    0.757677
1        0       35    0.781327
2        0       75    0.678469
3        0      114    0.616308
4        0      142    0.677929
5        0      204    0.795452
6        0      206    0.790767
7        0      210    0.652482
8        0      239    0.757665
9        0      246    0.647727


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv

# Load the saved node features and graph structure.
h0_df = pd.read_csv("sp500_h0_pca32.csv")
x = torch.tensor(h0_df.drop(columns=["Ticker"]).to_numpy(dtype=np.float32))
tickers = h0_df["Ticker"].tolist()

# Load edge index and weights.
edge_index_np = np.load("sp500_cosine_knn_edge_index.npy")
edge_weight_np = np.load("sp500_cosine_knn_edge_weight.npy")
edge_index = torch.tensor(edge_index_np, dtype=torch.long)
edge_weight = torch.tensor(edge_weight_np, dtype=torch.float32)

if x.ndim != 2:
    raise ValueError(f"Expected 2D node-feature matrix, got shape {tuple(x.shape)}")
if edge_index.ndim != 2 or edge_index.shape[0] != 2:
    raise ValueError(f"Expected edge_index shape [2, E], got {tuple(edge_index.shape)}")


#Define a simple 2-layer GAT network to process the graph and produce refined node embeddings that capture both the initial risk factor profiles and the relationships between companies based on their risk similarities.
# The GATConv layers will allow the model to learn attention weights for each edge, enabling it to focus on the most relevant neighbors when updating node representations.
# This model is a smoke test to verify that the graph structure and node features are correctly aligned and that we can successfully run a forward pass to get GAT embeddings without errors.

class GATNet(torch.nn.Module):
    def __init__(self, in_channels: int, hidden_channels: int = 64, out_channels: int = 32, heads: int = 4, dropout: float = 0.2):
        super().__init__()
        self.dropout = dropout
        self.gat1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout, add_self_loops=True)
        self.gat2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=dropout, add_self_loops=True)

    def forward(self, x, edge_index, return_attention: bool = False):
        x = F.dropout(x, p=self.dropout, training=self.training)
        if return_attention:
            x, attn1 = self.gat1(x, edge_index, return_attention_weights=True)
            x = F.elu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            x, attn2 = self.gat2(x, edge_index, return_attention_weights=True)
            return x, attn1, attn2
        x = self.gat1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.gat2(x, edge_index)
        return x

# Build the network and run a forward pass to get initial GAT node embeddings.
model = GATNet(in_channels=x.shape[1], hidden_channels=64, out_channels=32, heads=4, dropout=0.2)
model.eval()

with torch.no_grad():
    gat_embeddings, attn1, attn2 = model(x, edge_index, return_attention=True)

# Save outputs for downstream training / inspection.
np.save("sp500_gat_embeddings.npy", gat_embeddings.cpu().numpy())

print("Node feature matrix:", tuple(x.shape))
print("Edge index:", tuple(edge_index.shape))
print("GAT output embeddings:", tuple(gat_embeddings.shape))
print("Attention layer 1 weight shape:", tuple(attn1[1].shape) if isinstance(attn1, tuple) else "n/a")
print("Attention layer 2 weight shape:", tuple(attn2[1].shape) if isinstance(attn2, tuple) else "n/a")
print("First ticker:", tickers[0])
print("First embedding slice:", gat_embeddings[0, :8])

c:\Users\adesi\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Node feature matrix: (420, 32)
Edge index: (2, 6062)
GAT output embeddings: (420, 32)
Attention layer 1 weight shape: (6482, 4)
Attention layer 2 weight shape: (6482, 1)
First ticker: MMM
First embedding slice: tensor([ 0.0336, -0.0084, -0.0146,  0.0023, -0.0312, -0.0157,  0.0326, -0.0005])


## Train the GAT on sector labels

I’m the using GICS Sector as the supervised target because the graph already captures relationships between companies, and sector is a natural node-level label that lets the GAT learn useful attention patterns.

The goal here is to train the network so it can:

- learn which neighboring companies matter most,
- predict a sector for each company,
- and give us confidence scores we can inspect later.

I’m using a train/validation/test split so the reported scores are meaningful rather than just memorizing the training nodes.

In [12]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
from torch_geometric.nn import GATConv
import requests

# Why this setup:
# - GICS Sector gives us a real node label for supervised learning.
# - We train on node features h^(0) plus graph structure so the model can learn both text semantics and neighbor influence.
# - We do not pass edge weights here, because GAT learns its own attention coefficients from node features.

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)



def normalize_ticker(ticker: str) -> str:
    return str(ticker).strip().upper().replace(".", "-")

# Load the aligned node features and company metadata.
h0_df = pd.read_csv("sp500_h0_pca32.csv")
# Merge with metadata to get GICS Sector labels for each node, which will be our target classes for supervised training.
meta = get_sp500_data().copy()
# Normalize tickers in metadata to match those in h0_df for a successful merge.
meta["Ticker"] = meta["Symbol"].astype(str).apply(normalize_ticker)

# After the merge, we should have a "GICS Sector" column for each node. If any are missing, it indicates a mismatch between the node features and metadata, which would be a critical issue for training.
node_df = h0_df.merge(meta[["Ticker", "GICS Sector"]], on="Ticker", how="left")
if node_df["GICS Sector"].isna().any():
    missing_count = int(node_df["GICS Sector"].isna().sum())
    raise ValueError(f"Missing sector labels for {missing_count} nodes after merge.")

# Prepare training data: node features X, labels y, and edge_index for the GAT model.
X = torch.tensor(node_df.drop(columns=["Ticker", "GICS Sector"]).to_numpy(dtype=np.float32))
le = LabelEncoder()
# Encode GICS Sector labels as integers for classification.
y = torch.tensor(le.fit_transform(node_df["GICS Sector"]), dtype=torch.long)
num_classes = len(le.classes_)

edge_index_np = np.load("sp500_cosine_knn_edge_index.npy")
edge_index = torch.tensor(edge_index_np, dtype=torch.long)

# Create a clean train/val/test split over nodes.
all_idx = np.arange(len(node_df))
train_idx, temp_idx = train_test_split(all_idx, test_size=0.3, random_state=seed, stratify=y.numpy())
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=seed, stratify=y.numpy()[temp_idx])

train_mask = torch.zeros(len(node_df), dtype=torch.bool)
val_mask = torch.zeros(len(node_df), dtype=torch.bool)
test_mask = torch.zeros(len(node_df), dtype=torch.bool)
train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

#Define a GAT based classifier that takes the intial node features and graph structure to predict the GICS sector for each company
#The model consists of 2 GAtConv layers, followed by a linear classifier
#The first GAT layer has multiple heads to allow the model to learn different types of relationships between companies, while the second GAT layer aggregates these into a single representation for classification.
#We use simple regularisation techniques like dropout and early stopping to prevent overfitting, given the relatively small number of nodes and the complexity of the task.
class GATClassifier(nn.Module):
    def __init__(self, in_channels: int, hidden_channels: int, num_classes: int, heads: int = 4, dropout: float = 0.25):
        super().__init__()
        self.dropout = dropout
        self.gat1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout, add_self_loops=True)
        self.gat2 = GATConv(hidden_channels * heads, hidden_channels, heads=1, concat=False, dropout=dropout, add_self_loops=True)
        self.classifier = nn.Linear(hidden_channels, num_classes)

    def forward(self, x, edge_index, return_attention: bool = False):
        x = F.dropout(x, p=self.dropout, training=self.training)
        if return_attention:
            x, attn1 = self.gat1(x, edge_index, return_attention_weights=True)
            x = F.elu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            x, attn2 = self.gat2(x, edge_index, return_attention_weights=True)
            x = F.elu(x)
            logits = self.classifier(x)
            return logits, attn1, attn2

        x = self.gat1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.gat2(x, edge_index)
        x = F.elu(x)
        return self.classifier(x)

model = GATClassifier(in_channels=X.shape[1], hidden_channels=64, num_classes=num_classes, heads=4, dropout=0.25)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = nn.CrossEntropyLoss()

best_val_f1 = -1.0
best_state = None
patience = 20
patience_left = patience

for epoch in range(1, 201):
    model.train()
    optimizer.zero_grad()
    logits = model(X, edge_index)
    loss = criterion(logits[train_mask], y[train_mask])
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_logits = model(X, edge_index)
        val_pred = val_logits[val_mask].argmax(dim=-1).cpu().numpy()
        val_true = y[val_mask].cpu().numpy()
        val_f1 = f1_score(val_true, val_pred, average="macro")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        patience_left = patience
    else:
        patience_left -= 1

    if epoch % 20 == 0 or epoch == 1:
        train_pred = logits[train_mask].argmax(dim=-1).cpu().numpy()
        train_acc = accuracy_score(y[train_mask].cpu().numpy(), train_pred)
        print(f"Epoch {epoch:03d} | loss={loss.item():.4f} | train_acc={train_acc:.3f} | val_f1={val_f1:.3f}")

    if patience_left <= 0:
        print(f"Early stopping at epoch {epoch}.")
        break

if best_state is not None:
    model.load_state_dict(best_state)



C:\Users\adesi\AppData\Local\Temp\ipykernel_15720\3457739051.py:12: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


Epoch 001 | loss=2.3965 | train_acc=0.153 | val_f1=0.056
Epoch 020 | loss=1.1620 | train_acc=0.616 | val_f1=0.544
Epoch 040 | loss=0.9399 | train_acc=0.653 | val_f1=0.549
Early stopping at epoch 47.


In [13]:
# Evaluate on the test set using the best model state saved during training, and print a classification report to understand performance across different sectors.
# We also inspect the learned attention weights for a few test nodes to see which neighbors the model focused on when making predictions, which can provide insights into the relationships between companies based on their risk factor profiles.
model.eval()
with torch.no_grad():
    logits, attn1, attn2 = model(X, edge_index, return_attention=True)
    probs = F.softmax(logits, dim=-1)
    test_pred = logits[test_mask].argmax(dim=-1).cpu().numpy()
    test_true = y[test_mask].cpu().numpy()

# Compute and print test accuracy, macro F1 score, and a detailed classification report to evaluate the model's performance on the unseen test set.
accuracy = accuracy_score(test_true, test_pred)
f1 = f1_score(test_true, test_pred, average="macro")

print("\nTest accuracy:", round(accuracy, 4))
print("Test macro F1:", round(f1, 4))
print("\nClassification report:\n")
print(classification_report(test_true, test_pred, target_names=le.classes_))

# Show top prediction scores for a few test nodes.
print("\nTop prediction scores for the first 5 test nodes:")
for idx in test_idx[:5]:
    top_prob, top_class = probs[idx].max(dim=-1)
    print({
        "Ticker": node_df.loc[idx, "Ticker"],
        "TrueSector": node_df.loc[idx, "GICS Sector"],
        "PredSector": le.classes_[int(top_class)],
        "Confidence": round(float(top_prob), 4),
    })

# Inspect learned attention for one company.
focus_ticker = "MSFT" if "MSFT" in node_df["Ticker"].values else node_df.iloc[0]["Ticker"]
focus_idx = int(node_df.index[node_df["Ticker"] == focus_ticker][0])

attn_edge_index_1, attn_alpha_1 = attn1
src_nodes = attn_edge_index_1[0].cpu().numpy()
dst_nodes = attn_edge_index_1[1].cpu().numpy()
alpha_vals = attn_alpha_1.mean(dim=-1).cpu().numpy() if attn_alpha_1.ndim == 2 else attn_alpha_1.cpu().numpy().reshape(-1)

focus_mask = src_nodes == focus_idx
top_pos = np.argsort(-alpha_vals[focus_mask])[:5]
focus_neighbors = pd.DataFrame({
    "Source": [focus_ticker] * len(top_pos),
    "Neighbor": [node_df.loc[int(dst_nodes[focus_mask][p]), "Ticker"] for p in top_pos],
    "Attention": [float(alpha_vals[focus_mask][p]) for p in top_pos],
})

print(f"\nTop attention neighbors for {focus_ticker} (layer 1):")
print(focus_neighbors)

# Save prediction outputs for later analysis.
results_df = node_df[["Ticker", "GICS Sector"]].copy()
results_df["PredictedSector"] = le.inverse_transform(logits.argmax(dim=-1).cpu().numpy())
results_df["Confidence"] = probs.max(dim=-1).values.cpu().numpy()
results_df.to_csv("sp500_gat_predictions.csv", index=False)

print("\nSaved model outputs to sp500_gat_predictions.csv")


Test accuracy: 0.6032
Test macro F1: 0.5965

Classification report:

                        precision    recall  f1-score   support

Communication Services       0.67      0.67      0.67         3
Consumer Discretionary       1.00      0.29      0.44         7
      Consumer Staples       0.43      0.60      0.50         5
                Energy       0.75      1.00      0.86         3
            Financials       0.60      0.75      0.67         8
           Health Care       0.54      0.88      0.67         8
           Industrials       0.55      0.60      0.57        10
Information Technology       0.71      0.50      0.59        10
             Materials       0.00      0.00      0.00         3
           Real Estate       1.00      0.67      0.80         3
             Utilities       1.00      0.67      0.80         3

              accuracy                           0.60        63
             macro avg       0.66      0.60      0.60        63
          weighted avg       0.6

## Test the trained GAT and find NVIDIA's closest competitors

This cell does two things:

1. Tests the trained GAT on the held-out nodes so we know the model generalizes.
2. Ranks NVIDIA's nearest competitors by combining:
   - technology overlap from the original text-based node features, and
   - graph/market behavior from the trained GAT embeddings.

That gives a practical competitor list instead of a pure text-similarity list.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics.pairwise import cosine_similarity

# Reuse the trained model outputs already in memory.
# Why this works:
# - Technology overlap is captured by the original 32-dim h^(0) features.
# - Market behavior is approximated by the trained GAT's class probability profile.
# - Combining the two gives a more stable competitor ranking than either one alone.

if "logits" not in globals() or "probs" not in globals() or "node_df" not in globals():
    raise ValueError("Run the GAT training cell first so logits, probs, and node_df are available.")

# 1) Quick test on the held-out nodes.
test_pred = logits[test_mask].argmax(dim=-1).cpu().numpy()
test_true = y[test_mask].cpu().numpy()
test_acc = accuracy_score(test_true, test_pred)
test_f1 = f1_score(test_true, test_pred, average="macro")
print(f"Held-out test accuracy: {test_acc:.4f}")
print(f"Held-out macro F1: {test_f1:.4f}")

# 2) Build a competitor ranking for NVIDIA.
#    We score each company against NVIDIA using both text overlap and graph-learned behavior.
nvda_candidates = node_df.index[node_df["Ticker"] == "NVDA"].tolist()
if not nvda_candidates:
    raise ValueError("Could not find NVDA in node_df.")
nvda_idx = int(nvda_candidates[0])

# Technology overlap from h^(0) node features.
tech_sim = cosine_similarity(X.cpu().numpy(), X.cpu().numpy()[[nvda_idx]])[:, 0]

# Market behavior from the trained GAT's class probabilities.
behavior_sim = cosine_similarity(probs.cpu().numpy(), probs.cpu().numpy()[[nvda_idx]])[:, 0]

# Blend the two signals. Weights can be tuned based on validation performance or domain knowledge.
combined_score = 0.60 * tech_sim + 0.40 * behavior_sim

ranking_df = node_df[["Ticker", "GICS Sector"]].copy()
ranking_df["TechOverlap"] = tech_sim
ranking_df["MarketBehavior"] = behavior_sim
ranking_df["CombinedScore"] = combined_score
ranking_df = ranking_df[ranking_df["Ticker"] != "NVDA"].sort_values("CombinedScore", ascending=False)

top5 = ranking_df.head(5).reset_index(drop=True)
print("\nNVIDIA's Top 5 Competitors based on technology overlap and market behavior:")
print(top5)

# Save the ranking for later inspection.
top5.to_csv("nvda_top_5_competitors.csv", index=False)

print("\nScoring notes:")
print("- TechOverlap = cosine similarity in h^(0) text features")
print("- MarketBehavior = cosine similarity in the trained GAT prediction space")
print("- CombinedScore = 0.60 * TechOverlap + 0.40 * MarketBehavior")

Held-out test accuracy: 0.6032
Held-out macro F1: 0.5965

NVIDIA's Top 5 Competitors based on technology overlap and market behavior:
  Ticker             GICS Sector  TechOverlap  MarketBehavior  CombinedScore
0   PLTR  Information Technology     0.914436        0.981594       0.941299
1     MU  Information Technology     0.905568        0.989216       0.939027
2    TXN  Information Technology     0.894414        0.977318       0.927576
3     EW             Health Care     0.909596        0.930040       0.917773
4    OKE                  Energy     0.893421        0.945243       0.914150

Scoring notes:
- TechOverlap = cosine similarity in h^(0) text features
- MarketBehavior = cosine similarity in the trained GAT prediction space
- CombinedScore = 0.60 * TechOverlap + 0.40 * MarketBehavior


In [15]:
#Test for Microsoft (MSFT)
from sklearn.metrics.pairwise import cosine_similarity
msft_candidates = node_df.index[node_df["Ticker"] == "MSFT"].tolist()
if not msft_candidates:
    raise ValueError("Could not find MSFT in node_df.")
msft_idx = int(msft_candidates[0])
tech_sim_msft = cosine_similarity(X.cpu().numpy(), X.cpu().numpy()[[msft_idx]])[:, 0]
behavior_sim_msft = cosine_similarity(probs.cpu().numpy(), probs.cpu().numpy()[[msft_idx]])[:, 0]
combined_score_msft = 0.60 * tech_sim_msft + 0.40 * behavior_sim_msft
ranking_df_msft = node_df[["Ticker", "GICS Sector"]].copy()
ranking_df_msft["TechOverlap"] = tech_sim_msft
ranking_df_msft["MarketBehavior"] = behavior_sim_msft
ranking_df_msft["CombinedScore"] = combined_score_msft
ranking_df_msft = ranking_df_msft[ranking_df_msft["Ticker"] != "MSFT"].sort_values("CombinedScore", ascending=False)
top5_msft = ranking_df_msft.head(5).reset_index(drop=True)
print("\nMicrosoft's Top 5 Competitors based on technology overlap and market behavior:")
print(top5_msft)




Microsoft's Top 5 Competitors based on technology overlap and market behavior:
  Ticker             GICS Sector  TechOverlap  MarketBehavior  CombinedScore
0   POOL  Consumer Discretionary     0.908356        0.976052       0.935434
1    HSY        Consumer Staples     0.881304        0.962649       0.913842
2   ALLE             Industrials     0.890379        0.938242       0.909524
3   PYPL              Financials     0.877021        0.948761       0.905717
4    UNP             Industrials     0.867009        0.952267       0.901112


In [16]:
# Sensitivity analysis: how rankings change as we tune tech vs behavior weights.
import numpy as np
import pandas as pd

if "node_df" not in globals() or "X" not in globals() or "probs" not in globals():
    raise ValueError("Run the training/evaluation cells first so node_df, X, and probs are available.")

anchor = "MSFT"
anchor_candidates = node_df.index[node_df["Ticker"] == anchor].tolist()
if not anchor_candidates:
    raise ValueError(f"Could not find {anchor} in node_df.")
anchor_idx = int(anchor_candidates[0])

tech_sim = cosine_similarity(X.cpu().numpy(), X.cpu().numpy()[[anchor_idx]])[:, 0]
behavior_sim = cosine_similarity(probs.cpu().numpy(), probs.cpu().numpy()[[anchor_idx]])[:, 0]

# Put both signals on comparable [0, 1] scales for fair weight tuning.
def minmax(v: np.ndarray) -> np.ndarray:
    vmin, vmax = float(v.min()), float(v.max())
    return (v - vmin) / (vmax - vmin + 1e-12)

tech_scaled = minmax(tech_sim)
behavior_scaled = minmax(behavior_sim)

print(f"{anchor} signal ranges (raw cosine):")
print(f"  TechOverlap   min={tech_sim.min():.4f}, max={tech_sim.max():.4f}")
print(f"  MarketBehavior min={behavior_sim.min():.4f}, max={behavior_sim.max():.4f}")

weight_grid = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
summary_rows = []

for w_tech in weight_grid:
    w_behavior = 1.0 - w_tech
    combined = w_tech * tech_scaled + w_behavior * behavior_scaled

    ranking = node_df[["Ticker", "GICS Sector"]].copy()
    ranking["TechOverlap"] = tech_sim
    ranking["MarketBehavior"] = behavior_sim
    ranking["CombinedScore"] = combined
    ranking = ranking[ranking["Ticker"] != anchor].sort_values("CombinedScore", ascending=False).reset_index(drop=True)

    top5 = ranking.head(5)
    summary_rows.append({
        "w_tech": w_tech,
        "w_behavior": w_behavior,
        "Top5": ", ".join(top5["Ticker"].tolist()),
        "Top1": top5.iloc[0]["Ticker"],
        "Top1Sector": top5.iloc[0]["GICS Sector"],
    })

summary_df = pd.DataFrame(summary_rows)
print("\nTop competitors as weights change (scaled-score blend):")
print(summary_df)

# Optional: compare with your current fixed weights 0.60 / 0.40 (without scaling).
current_combined_raw = 0.60 * tech_sim + 0.40 * behavior_sim
current_ranking_raw = node_df[["Ticker", "GICS Sector"]].copy()
current_ranking_raw["TechOverlap"] = tech_sim
current_ranking_raw["MarketBehavior"] = behavior_sim
current_ranking_raw["CombinedScore"] = current_combined_raw
current_ranking_raw = current_ranking_raw[current_ranking_raw["Ticker"] != anchor].sort_values("CombinedScore", ascending=False)

print("\nCurrent method (raw cosine, w_tech=0.60, w_behavior=0.40):")
print(current_ranking_raw.head(5).reset_index(drop=True))

MSFT signal ranges (raw cosine):
  TechOverlap   min=-0.8821, max=1.0000
  MarketBehavior min=0.0246, max=1.0000

Top competitors as weights change (scaled-score blend):
   w_tech  w_behavior                        Top5  Top1  \
0     0.0         1.0    POOL, TTWO, HSY, PH, UNP  POOL   
1     0.2         0.8  POOL, TTWO, HSY, UNP, PYPL  POOL   
2     0.4         0.6  POOL, HSY, TTWO, UNP, PYPL  POOL   
3     0.6         0.4  POOL, HSY, PYPL, ALLE, UNP  POOL   
4     0.8         0.2  POOL, HSY, ALLE, PYPL, LNT  POOL   
5     1.0         0.0  VRSN, POOL, ALLE, ACN, COF  VRSN   

               Top1Sector  
0  Consumer Discretionary  
1  Consumer Discretionary  
2  Consumer Discretionary  
3  Consumer Discretionary  
4  Consumer Discretionary  
5  Information Technology  

Current method (raw cosine, w_tech=0.60, w_behavior=0.40):
  Ticker             GICS Sector  TechOverlap  MarketBehavior  CombinedScore
0   POOL  Consumer Discretionary     0.908356        0.976052       0.935434
1    H

In [17]:
# Export precomputed embeddings for high-speed API serving (no PyTorch inference at runtime).
import numpy as np
import pandas as pd

if "node_df" not in globals() or "X" not in globals() or "probs" not in globals():
    raise ValueError("Run training/eval cells first so node_df, X, probs are available.")

# 1) Save exact vectors used by ranking logic.
tech_embeddings = X.detach().cpu().numpy().astype(np.float32)          # h^(0) / tech space
behavior_embeddings = probs.detach().cpu().numpy().astype(np.float32)  # behavior space

np.save("tech_embeddings.npy", tech_embeddings)
np.save("behavior_embeddings.npy", behavior_embeddings)

# 2) Save ticker metadata for index lookup.
lookup_df = node_df[["Ticker", "GICS Sector"]].copy().reset_index(drop=True)
lookup_df.to_csv("ticker_lookup.csv", index=False)

print("Saved precomputed ranking artifacts:")
print(f"- tech_embeddings.npy: {tech_embeddings.shape}")
print(f"- behavior_embeddings.npy: {behavior_embeddings.shape}")
print(f"- ticker_lookup.csv: {lookup_df.shape}")

Saved precomputed ranking artifacts:
- tech_embeddings.npy: (420, 32)
- behavior_embeddings.npy: (420, 11)
- ticker_lookup.csv: (420, 2)


In [2]:
import os
import torch
import torch.onnx

# Save the trained model in PyTorch format (.pt).
pytorch_model_path = "sp500_gat_model.pt"
torch.save(model.state_dict(), pytorch_model_path)

# Check file size in MB.
pytorch_size_bytes = os.path.getsize(pytorch_model_path)
pytorch_size_mb = pytorch_size_bytes / (1024 * 1024)

print(f"PyTorch model saved to {pytorch_model_path}")
print(f"PyTorch model size: {pytorch_size_bytes:,} bytes ({pytorch_size_mb:.3f} MB)")


PyTorch model saved to sp500_gat_model.pt
PyTorch model size: 108,904 bytes (0.104 MB)


In [3]:
import torch.onnx

# Prepare dummy inputs for ONNX export.
# The model takes node features X and edge_index as inputs.
dummy_x = torch.randn(X.shape[0], X.shape[1], dtype=torch.float32)
dummy_edge_index = torch.zeros(2, edge_index.shape[1], dtype=torch.long)

# Export to ONNX format.
onnx_model_path = "sp500_gat_model.onnx"

try:
    torch.onnx.export(
        model,
        (dummy_x, dummy_edge_index),
        onnx_model_path,
        input_names=["node_features", "edge_index"],
        output_names=["logits"],
        dynamic_axes={
            "node_features": {0: "num_nodes"},
            "edge_index": {1: "num_edges"},
            "logits": {0: "num_nodes"},
        },
        opset_version=12,
        do_constant_folding=True,
    )
    
    onnx_size_bytes = os.path.getsize(onnx_model_path)
    onnx_size_mb = onnx_size_bytes / (1024 * 1024)
    
    print(f"ONNX model saved to {onnx_model_path}")
    print(f"ONNX model size: {onnx_size_bytes:,} bytes ({onnx_size_mb:.3f} MB)")
    
    # Size comparison.
    ratio = onnx_size_mb / pytorch_size_mb
    diff = pytorch_size_mb - onnx_size_mb
    print(f"\n--- Size Comparison ---")
    print(f"PyTorch: {pytorch_size_mb:.3f} MB")
    print(f"ONNX:    {onnx_size_mb:.3f} MB")
    print(f"Difference: {diff:+.3f} MB ({(ratio-1)*100:+.1f}%)")
    if onnx_size_mb < pytorch_size_mb:
        print(f"ONNX is {ratio:.2f}x smaller")
    else:
        print(f"ONNX is {1/ratio:.2f}x larger")
        
except Exception as e:
    print(f"ONNX export failed (may be due to GATConv operator support):")
    print(f"Error: {e}")
    print("Note: ONNXRuntime doesn't natively support GATConv; you may need to:")
    print("  1. Use a third-party converter (e.g., skl2onnx, torch2onnx)")
    print("  2. Implement GATConv as a custom ONNX operator")
    print("  3. Export just the classifier head + pre-computed embeddings")


ONNX export failed (may be due to GATConv operator support):
Error: Exporting the operator 'aten::scatter_reduce' to ONNX opset version 12 is not supported. Support for this operator was added in version 16, try exporting with this version.
Note: ONNXRuntime doesn't natively support GATConv; you may need to:
  1. Use a third-party converter (e.g., skl2onnx, torch2onnx)
  2. Implement GATConv as a custom ONNX operator
  3. Export just the classifier head + pre-computed embeddings


C:\Users\adesi\AppData\Local\Temp\ipykernel_10500\3188906480.py:76: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if return_attention:
c:\Users\adesi\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\onnx\symbolic_opset9.py:5333: UserWarning: Exporting aten::index operator with indices of type Byte. Only 1-D indices are supported. In any other case, this will produce an incorrect ONNX graph.
  warnings.warn(


In [3]:
# Export just the classifier head as a lightweight ONNX model.
import onnx  # Import explicitly to ensure it's available
import torch.onnx

dummy_embeddings = torch.randn(1, 64, dtype=torch.float32)

classifier_onnx_path = "sp500_gat_classifier_head.onnx"

try:
    torch.onnx.export(
        model.classifier,
        dummy_embeddings,
        classifier_onnx_path,
        input_names=["embeddings"],
        output_names=["logits"],
        dynamic_axes={
            "embeddings": {0: "num_nodes"},
            "logits": {0: "num_nodes"},
        },
        opset_version=13,
        do_constant_folding=True,
        verbose=False,
    )
    
    classifier_size_bytes = os.path.getsize(classifier_onnx_path)
    classifier_size_mb = classifier_size_bytes / (1024 * 1024)
    
    print(f"✓ Classifier head (ONNX) saved to {classifier_onnx_path}")
    print(f"  Size: {classifier_size_bytes:,} bytes ({classifier_size_mb:.4f} MB)")
    print(f"\n--- Model Size Comparison ---")
    print(f"PyTorch (full GAT):        {pytorch_size_mb:.4f} MB")
    print(f"ONNX (classifier only):    {classifier_size_mb:.4f} MB")
    print(f"Ratio:                     {classifier_size_mb/pytorch_size_mb*100:.1f}%")
    print(f"\nNote: Classifier head is much smaller because it's just a linear layer.")
    print(f"Full GAT model can't export to ONNX (needs opset 16+ for GATConv).")
    
except Exception as e:
    print(f"✗ Classifier head export failed: {e}")

✓ Classifier head (ONNX) saved to sp500_gat_classifier_head.onnx
  Size: 3,102 bytes (0.0030 MB)

--- Model Size Comparison ---
PyTorch (full GAT):        0.1039 MB
ONNX (classifier only):    0.0030 MB
Ratio:                     2.8%

Note: Classifier head is much smaller because it's just a linear layer.
Full GAT model can't export to ONNX (needs opset 16+ for GATConv).


In [5]:
print("=" * 60)
print("MODEL EXPORT SUMMARY")
print("=" * 60)
print("\n✓ PyTorch Format (.pt)")
print(f"  - File: sp500_gat_model.pt")
print(f"  - Size: {pytorch_size_mb:.4f} MB ({pytorch_size_bytes:,} bytes)")
print(f"  - Contains: Full GAT classifier with gat1, gat2, classifier layers")
print(f"  - Use case: Training, research, local inference")

print("\n✓ ONNX Format (.onnx) - Classifier Head Only")
print(f"  - File: sp500_gat_classifier_head.onnx")
print(f"  - Size: {classifier_size_mb:.4f} MB ({classifier_size_bytes:,} bytes)")
print(f"  - Contains: Linear classification layer only")
print(f"  - Size reduction: {(1 - classifier_size_mb/pytorch_size_mb)*100:.1f}% smaller")
print(f"  - Use case: Production inference, cross-platform deployment")

print("\n⚠ Full GAT → ONNX Conversion Limitation")
print(f"  - PyTorch GATConv uses scatter_reduce (needs ONNX opset 16+)")
print(f"  - We exported only opset 13 (standard)")
print(f"  - Solution: Split pipeline into two stages:")
print(f"    Stage 1 (PyTorch): Extract GAT embeddings")
print(f"    Stage 2 (ONNX): Classify embeddings")

print("\n📋 Recommended Production Workflow")
print(f"  1. Load full model (PyTorch): classifier = GATClassifier(...)")
print(f"  2. Extract embeddings (PyTorch): embeddings = model(X, edge_index)")
print(f"  3. Classify with ONNX Runtime: probs = onnx_session.run(embeddings)")
print(f"  4. Benefits: Portable, ~97% smaller, faster inference")

print("\n" + "=" * 60)
print("Files saved in: c:\\Users\\adesi\\Desktop\\GraphS&P500\\")
print("=" * 60)

MODEL EXPORT SUMMARY

✓ PyTorch Format (.pt)
  - File: sp500_gat_model.pt
  - Size: 0.1039 MB (108,904 bytes)
  - Contains: Full GAT classifier with gat1, gat2, classifier layers
  - Use case: Training, research, local inference

✓ ONNX Format (.onnx) - Classifier Head Only
  - File: sp500_gat_classifier_head.onnx
  - Size: 0.0030 MB (3,102 bytes)
  - Contains: Linear classification layer only
  - Size reduction: 97.2% smaller
  - Use case: Production inference, cross-platform deployment

⚠ Full GAT → ONNX Conversion Limitation
  - PyTorch GATConv uses scatter_reduce (needs ONNX opset 16+)
  - We exported only opset 13 (standard)
  - Solution: Split pipeline into two stages:
    Stage 1 (PyTorch): Extract GAT embeddings
    Stage 2 (ONNX): Classify embeddings

📋 Recommended Production Workflow
  1. Load full model (PyTorch): classifier = GATClassifier(...)
  2. Extract embeddings (PyTorch): embeddings = model(X, edge_index)
  3. Classify with ONNX Runtime: probs = onnx_session.run(embe